In [ ]:
!pip -q install -U huggingface_hub

import os, json, collections
from huggingface_hub import hf_hub_download
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/miracl_eval"
os.makedirs(OUT, exist_ok=True)

# where your indexes are (needed to check which judged passages exist in YOUR corpus)
PACKS = {
    "en": "/content/drive/MyDrive/miracl_en_indexes",
    "hi": "/content/drive/MyDrive/miracl_hi_indexes",
}
SPLIT = "dev"     # MIRACL's standard evaluation split (the test splits have no public judgments)

def fetch(lang, kind, split):
    # kind = "topics" or "qrels"
    fn = f"miracl-v1.0-{lang}/{kind}/{kind}.miracl-v1.0-{lang}-{split}.tsv"
    return hf_hub_download("miracl/miracl", fn, repo_type="dataset")

def load_topics(path):
    q = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            qid, text = line.rstrip("\n").split("\t", 1)
            q[qid] = text
    return q

def load_qrels(path):
    # lines: query_id  Q0  docid  relevance(0/1)
    qrels = collections.defaultdict(dict)
    with open(path, encoding="utf-8") as f:
        for line in f:
            qid, _, docid, rel = line.split()
            qrels[qid][docid] = int(rel)
    return dict(qrels)

for lang, pack in PACKS.items():
    queries = load_topics(fetch(lang, "topics", SPLIT))
    qrels   = load_qrels(fetch(lang, "qrels", SPLIT))

    # --- keep only judgments for passages that exist in YOUR index ---
    indexed = set(json.load(open(f"{pack}/docids.json")))
    eval_qrels, eval_queries = {}, {}
    for qid, judged in qrels.items():
        kept = {d: r for d, r in judged.items() if d in indexed}
        if any(r > 0 for r in kept.values()):          # must have >=1 relevant passage in the index
            eval_qrels[qid] = kept
            eval_queries[qid] = queries[qid]

    # --- coverage report ---
    pos_all  = sum(r > 0 for j in qrels.values() for r in j.values())
    pos_kept = sum(r > 0 for j in eval_qrels.values() for r in j.values())
    print(f"[{lang}] {SPLIT}: {len(queries)} queries | "
          f"{len(eval_queries)} usable | relevant passages in index: {pos_kept}/{pos_all} "
          f"({100*pos_kept/pos_all:.1f}%)")

    # --- save ---
    d = f"{OUT}/{lang}"
    os.makedirs(d, exist_ok=True)
    json.dump(queries,      open(f"{d}/queries_{SPLIT}_full.json", "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(qrels,        open(f"{d}/qrels_{SPLIT}_full.json", "w"))
    json.dump(eval_queries, open(f"{d}/eval_queries.json", "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(eval_qrels,   open(f"{d}/eval_qrels.json", "w"))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.3.0 which is incompatible.
Mounted at /content/drive


topics.miracl-v1.0-en-dev.tsv:   0%|          | 0.00/36.8k [00:00<?, ?B/s]

qrels.miracl-v1.0-en-dev.tsv:   0%|          | 0.00/168k [00:00<?, ?B/s]

[en] dev: 799 queries | 230 usable | relevant passages in index: 333/2326 (14.3%)


topics.miracl-v1.0-hi-dev.tsv:   0%|          | 0.00/52.0k [00:00<?, ?B/s]

qrels.miracl-v1.0-hi-dev.tsv:   0%|          | 0.00/79.8k [00:00<?, ?B/s]

[hi] dev: 350 queries | 350 usable | relevant passages in index: 752/752 (100.0%)


In [ ]:
import json, itertools
from google.colab import drive
drive.mount("/content/drive")

LANG = "hi"      # "en" or "hi"
E = f"/content/drive/MyDrive/miracl_eval/{LANG}"
P = f"/content/drive/MyDrive/miracl_{LANG}_indexes/passages.jsonl"

queries = json.load(open(f"{E}/eval_queries.json", encoding="utf-8"))
qrels   = json.load(open(f"{E}/eval_qrels.json"))

# 1. Overview
print(f"{len(queries)} queries, {len(qrels)} with judgments")
n_rel = [sum(r > 0 for r in j.values()) for j in qrels.values()]
n_non = [sum(r == 0 for r in j.values()) for j in qrels.values()]
print(f"relevant per query: avg {sum(n_rel)/len(n_rel):.1f}, max {max(n_rel)}")
print(f"judged non-relevant per query: avg {sum(n_non)/len(n_non):.1f}")

# 2. First few queries
print("\n--- QUERIES (qid -> text) ---")
for qid, text in itertools.islice(queries.items(), 5):
    print(qid, "->", text)

# 3. Qrels for those queries
print("\n--- QRELS (qid -> {docid: relevance}) ---")
for qid in itertools.islice(queries, 3):
    print(qid, "->", dict(itertools.islice(qrels[qid].items(), 6)))

# 4. One query with the actual passage text
qid = next(iter(queries))
wanted = set(qrels[qid])
texts = {}
with open(P, encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        if d["id"] in wanted:
            texts[d["id"]] = d
            if len(texts) == len(wanted): break

print(f"\n=== QUERY {qid}: {queries[qid]} ===")
for docid, rel in qrels[qid].items():
    d = texts.get(docid)
    if d:
        label = "RELEVANT    " if rel else "NOT RELEVANT"
        print(f"\n[{label}] {docid} | {d['title']}\n  {d['text'][:250]}...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
350 queries, 350 with judgments
relevant per query: avg 2.1, max 9
judged non-relevant per query: avg 7.8

--- QUERIES (qid -> text) ---
1004960#0 -> किस यंत्र में मापन के समय अधिकतम पावर व्यय होती है?
1006614#0 -> अल्जीरिया किस मरुस्थल के अंतर्गत स्थित है?
100764#0 -> च्चतम न्यायालय के न्यायाधीशों की संख्या में वृद्धि करने की शक्ति किसके पास है?
1008914#0 -> आम तौर पर सोडियम बाइकार्बोनेट को क्या कहा जाता है?
100955#0 -> स्वर्णिम चतुभरुज योजना किससे संबंधित है?

--- QRELS (qid -> {docid: relevance}) ---
1004960#0 -> {'69380#11': 1, '555530#0': 1, '555530#1': 0, '432721#1': 0, '496800#30': 0, '174163#10': 0}
1006614#0 -> {'8907#24': 1, '8907#1': 1, '8907#26': 1, '8907#21': 1, '166718#0': 0, '3261#0': 0}
100764#0 -> {'30472#2': 1, '30472#7': 0, '10069#4': 0, '247208#8': 0, '247208#9': 0, '30472#11': 0}

=== QUERY 1004960#0: किस यंत्र में मापन के समय अधिकतम पावर

In [ ]:
import json

for lang in ["en", "hi"]:
    q = json.load(open(f"/content/drive/MyDrive/miracl_eval/{lang}/eval_queries.json", encoding="utf-8"))
    print(f"\n{'=' * 20} {lang.upper()}: {len(q)} queries {'=' * 20}")
    for i, (qid, text) in enumerate(q.items(), 1):
        print(f"{i:>4}. {qid} | {text}")


==================== EN: 230 queries ====================
   1. 0 | Is Creole a pidgin of French?
   2. 20 | What percentage of the Earth's atmosphere is oxygen?
   3. 34 | Why is it called guerrilla?
   4. 35 | What was the first film directed by Andrei Tarkovsky?
   5. 47 | When did Aristagoras become leader of Miletus?
   6. 61 | Where are polycyclic aromatic hydrocarbons found?
   7. 119 | How large is Bengal by area?
   8. 164 | Is Indonesia a democracy?
   9. 174 | What is the difference between a telenovela and a soap opera?
  10. 177 | How many provinces are in Mongolia?
  11. 238 | What is the most used language in Europe?
  12. 337 | When was the Commonwealth of Independent States formed?
  13. 338 | When was the first Gameboy released by Nintendo?
  14. 359 | What is the capital of Angola?
  15. 366 | What is the U.S. National Debt?
  16. 430 | Who first identified blood types?
  17. 467 | What is an Umayyad caliph?
  18. 517 | When was Syracuse founded by the Greeks?
  19.

In [ ]:
import json

BASE = "/content/drive/MyDrive/miracl_eval"

for lang in ["en", "hi"]:
    full = json.load(open(f"{BASE}/{lang}/queries_dev_full.json", encoding="utf-8"))
    usable = json.load(open(f"{BASE}/{lang}/eval_queries.json", encoding="utf-8"))
    print(f"\n{'=' * 20} {lang.upper()}: {len(full)} total, {len(usable)} usable in current index {'=' * 20}")
    for i, (qid, text) in enumerate(full.items(), 1):
        flag = " " if qid in usable else "x"      # x = no relevant passage in your current index
        print(f"{i:>4}. [{flag}] {qid} | {text}")


==================== EN: 799 total, 230 usable in current index ====================
   1. [ ] 0 | Is Creole a pidgin of French?
   2. [ ] 20 | What percentage of the Earth's atmosphere is oxygen?
   3. [x] 32 | When did Marxism develop?
   4. [x] 33 | Which Assassin's Creed is the latest released?
   5. [ ] 34 | Why is it called guerrilla?
   6. [ ] 35 | What was the first film directed by Andrei Tarkovsky?
   7. [x] 37 | Who wrote the song "Happy Days"?
   8. [x] 43 | What was the first Atlantic hurricane in 2000?
   9. [x] 44 | What is the main version of Islam practiced in Iraq?
  10. [ ] 47 | When did Aristagoras become leader of Miletus?
  11. [x] 49 | What is Captain Cold's real name?
  12. [ ] 61 | Where are polycyclic aromatic hydrocarbons found?
  13. [x] 73 | Did the Stackhouse books get made into a show?
  14. [x] 79 | Where are Porsches manufactured?
  15. [x] 98 | What is the population of Lagos?
  16. [x] 100 | How old is the oldest pyramid?
  17. [x] 105 | Is Abi Brann

In [ ]:
import json, os
from google.colab import drive
drive.mount("/content/drive")

BASE  = "/content/drive/MyDrive/miracl_eval"
PACKS = {"en": "/content/drive/MyDrive/miracl_en_indexes",
         "hi": "/content/drive/MyDrive/miracl_hi_indexes"}

HARD = {
  "en": {
    "negation": {
      "2710": "Are all streams rivers?",
      "2879": "Do praying mantises always eat their mates?",
      "1839": "Has Catholicism always been the national religion of Spain?",
      "3610": "Is Welsh a dialect of English?",
      "2845": "Is Zeus a Titan?",
      "1763": "Is Genoa a country?",
      "1665": "Is Fortnite an multiplayer online role-playing game?",
    },
    "homonym_ambiguity": {
      "901":  "What is a Cricket bat called?",
      "672":  "Who is the CEO of Ram?",
      "3932": "What is a ray in geometry?",
      "2773": "Where did Rugby originate?",
      "1253": "When was Cromwell executed?",
      "2323": "Who won the Battle of Vienna?",
      "1915": "Where is Campbelltown located?",
    },
    "temporal": {
      "3314": "What came before the Bronze Age?",
      "3800": "What was the eon before the Phanerozoic Eon?",
      "2024": "Who was the last ruling monarch of Spain?",
      "2021": "When did the Enlightenment spread to America?",
      "3837": "Was Ireland ever part of the UK?",
      "2612": "When did France join World War I?",
    },
  },
  "hi": {
    "negation": {
      "51793#0":  "जैन धर्म के लोग प्याज क्यों नही खाते?",
      "23511#0":  "यूरोप के अलावा, कौन सा एकमात्र महाद्वीप है जिसमें से देशों ने पुरुषों के लिए फुटबॉल विश्व कप जीता है?",
      "596559#0": "घर्षण द्वारा किए गए कार्य का क्षय नहीं होता है बल्कि यह ऊर्जा में रूपांतरित हो जाती है?",
      "902061#0": "अगर निर्मला सीतारमण भारत में रक्षा मंत्रालय के पोर्टफोलियो को बनाए रखने वाली दूसरी महिला बन गई, तो सबसे पहले कौन था?",
      "230675#0": "भारत के राष्ट्रवादी नेताओं ने साइमन कमीशन का बहिष्कार किया था, क्योंकि?",
      "1177052#0":"कितनी बार एक वर्ष में कम से कम संसद की बैठक होना आवश्यक है?",
      "625160#0": "किस राज्य ने राज्य में शराब का सेवन पूरी बंद कर दिया है?",
    },
    "homonym_ambiguity": {
      "1240100#0":"क्रिकेट नाम किस प्रसिद्ध ब्रिटिश व्यक्ति ने दिया था?",
      "182473#0": "मंगल के २ उपग्रह के क्या नाम है?",
      "1346725#0":"शिवाजी द बॉस किसकी फिल्म है?",
      "1033752#0":"कांग्रेस दल का नेता कौन है ?",
      "674967#0": "राजा हरिश्चंद्र फिल्म को किस भारतीय फिल्म द्वारा निर्मित किया गया था?",
      "874129#0": "मुद्रास्फीति के दौरान कर की दरों पर क्या प्रभाव पड़ता है?",
      "1281069#0":"अमेरिका की खोज किसने की?",
    },
    "temporal": {
      "433508#0": "चरखी दादरी का नाम सन् 1939 में डालमिया दादरी रखा गया, इसका नाम पुनः चरखी दादरी कब पड़ा?",
      "1174018#0":"1947 में भारत के स्वतंत्र होने के पश्चात तक हरियाणा किस प्रान्त का भाग था?",
      "23966#0":  "पंजाब के मुख्यमंत्री कैप्टन अमरिंदर सिंह के इस्तीफा के बाद किसने पंजाब के नए मुख्यमंत्री के रूप में शपथ ली?",
      "703838#0": "चौरी-चौरा कांड के बाद महात्मा गाँधी ने कौनसा आन्दोलन स्थगित कर दिया था?",
      "229856#0": "वर्ष १९८५ में भारत का प्रधानमंत्री कौन था?",
      "229953#0": "११९१ ई. में तराइन का युद्ध मोहम्मद गोरी और किसके बीच हुआ था?",
    },
  },
}

import unicodedata, re

def norm(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", s)).strip()

for lang, cats in HARD.items():
    d = f"{BASE}/{lang}"
    all_q = json.load(open(f"{d}/queries_dev_full.json", encoding="utf-8"))
    all_r = json.load(open(f"{d}/qrels_dev_full.json"))
    indexed = set(json.load(open(f"{PACKS[lang]}/docids.json")))

    queries, qrels_full, qrels_idx, meta, usable = {}, {}, {}, {}, []
    for cat, items in cats.items():
        for qid, text in items.items():
            assert qid in all_q, f"[{lang}] {qid} not in dev queries"      # the ID must exist
            if norm(all_q[qid]) != norm(text):                              # text differences only warn
                print(f"  WARNING [{lang}] {qid}: pasted text differs from dataset.\n"
                      f"    dataset: {all_q[qid]}\n    pasted : {text}")
            queries[qid]    = all_q[qid]                                    # always use the dataset's text
            qrels_full[qid] = all_r[qid]
            qrels_idx[qid]  = {doc: r for doc, r in all_r[qid].items() if doc in indexed}
            meta[qid]       = cat
            if any(r > 0 for r in qrels_idx[qid].values()):
                usable.append(qid)

    json.dump(queries,    open(f"{d}/hard_queries.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    json.dump(qrels_full, open(f"{d}/hard_qrels_full.json", "w"))
    json.dump(qrels_idx,  open(f"{d}/hard_qrels.json", "w"))
    json.dump(meta,       open(f"{d}/hard_meta.json", "w"))
    json.dump(usable,     open(f"{d}/hard_usable.json", "w"))

    print(f"\n[{lang}] saved {len(queries)} hard queries | usable in current index: {len(usable)}/{len(queries)}")
    for qid in queries:
        n_full = sum(r > 0 for r in qrels_full[qid].values())
        n_idx  = sum(r > 0 for r in qrels_idx[qid].values())
        print(f"  {'OK     ' if qid in usable else 'MISSING'}  {meta[qid]:<18} {qid:<10} relevant in index {n_idx}/{n_full}  | {queries[qid][:60]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[en] saved 20 hard queries | usable in current index: 9/20
  MISSING  negation           2710       relevant in index 0/5  | Are all streams rivers?
  MISSING  negation           2879       relevant in index 0/1  | Do praying mantises always eat their mates?
  OK       negation           1839       relevant in index 1/5  | Has Catholicism always been the national religion of Spain?
  OK       negation           3610       relevant in index 2/9  | Is Welsh a dialect of English?
  OK       negation           2845       relevant in index 1/2  | Is Zeus a Titan?
  MISSING  negation           1763       relevant in index 0/2  | Is Genoa a country?
  MISSING  negation           1665       relevant in index 0/2  | Is Fortnite an multiplayer online role-playing game?
  MISSING  homonym_ambiguity  901        relevant in index 0/1  | What is a Cricket bat called?
  MI